In [1]:

from dataloader import create_dataloaders, load_noisy_dataset_by_task
from lora_model_alpha import LORAEngine
from influence import IFEngine

from tqdm import tqdm
import pickle as pkl
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    
    
    
    
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoModelForCausalLM
)
import re

import numpy as np



/home/haskari/miniconda3/envs/alphalora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_layer_number(s):
    match = re.search(r'\.(\d+)\.', s)
    return match.group(0) if match else None


base_path = "mistralai/Mistral-7B-v0.1"     #"mistralai/Mistral-7B-v0.1"    #"google/gemma-7b"    

base_model = AutoModelForCausalLM.from_pretrained(
    base_path,
    #quantization_config=quantization_config,
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    offload_folder="offload",
    offload_state_dict=True,
    device_map='auto'
)
layers={}
for k,v in base_model.named_parameters():
    #print(k)
    result = extract_layer_number(k)
    layers[result]=0

layers=list(layers.keys())

layers.remove(None)

# IFs=[]

# for layer in layers:
#     with open(f'results_{layer}.pkl', 'rb') as f:
#         IF_pickle=pkl.load(f)
#     IFs.append(IF_pickle['influence']['proposed'].to_numpy().sum())

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.25s/it]


In [ ]:
# def scale_values_final(values, target_sum, exponent=1.5): #OLD IMPLEMENTATION
#     values = np.array(values, dtype=np.float64)  # Ensure numerical stability

#     # Invert values so less negative numbers remain smaller after transformation
#     inverted_values = np.max(values) - values  
    
#     # Avoid division by zero in case all values are identical
#     if np.all(inverted_values == 0):
#         inverted_values += 1e-6
    
#     scaled_values = np.power(inverted_values, exponent)

#     # Normalize to sum to target_sum
#     total_scaled = scaled_values.sum()
#     if total_scaled == 0:
#         raise ValueError("Scaling resulted in zero sum, check input values.")
    
#     # Convert scaled values into integer distribution while keeping sum == target_sum
#     scaled_fractions = (scaled_values / total_scaled) * target_sum
#     scaled_integers = np.round(scaled_fractions).astype(int)
    
#     # Ensure no values are zero (minimum value of 1)
#     scaled_integers = np.maximum(scaled_integers, 1)

#     # Adjust sum to match target_sum using the difference approach
#     while scaled_integers.sum() != target_sum:
#         difference = target_sum - scaled_integers.sum()
#         # print(scaled_integers.sum())

#         # Compute differences to decide where to adjust
#         adjustment_values = scaled_values - scaled_integers

#         if difference > 0:
#             idx = np.argmin(adjustment_values)  # Find smallest difference and increase it
#             scaled_integers[idx] += 1
#         else:
#             idx = np.argmax(adjustment_values)  # Find largest difference and decrease it
#             if scaled_integers[idx] > 1:
#                 scaled_integers[idx] -= 1  # Ensure minimum value remains 1

#     return scaled_integers

In [3]:
def scale_values_final(values, target_sum, exponent=1.5):
    values = np.array(values, dtype=np.float64)
    n = len(values)

    # Invert values so smaller (less negative) ones get smaller weights
    inverted_values = np.max(values) - values
    if np.all(inverted_values == 0):
        inverted_values += 1e-6

    scaled_values = np.power(inverted_values, exponent)
    scaled_fractions = (scaled_values / scaled_values.sum()) * (target_sum - n)  # Subtract 1 per value

    # Floor and add the minimum value of 1
    floored = np.floor(scaled_fractions).astype(int) + 1
    remainder = target_sum - floored.sum()

    # Distribute remaining units to values with largest fractional parts
    fractional_parts = scaled_fractions - np.floor(scaled_fractions)
    top_indices = np.argsort(fractional_parts)[::-1]

    for i in range(remainder):
        floored[top_indices[i]] += 1

    return floored

In [12]:
check=[2, 6, 6, 6, 6, 5, 7, 6, 8, 7, 6, 7, 6, 6, 6, 5,
 6, 6, 5, 4, 5, 5, 4, 2, 4, 5, 3, 4, 2, 3, 4, 3]
sum(check)

160

In [13]:
# names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']
# for name in names:
#     IFs = []
#     for layer in layers:
#         with open(f'results_{layer}_{name}.pkl', 'rb') as f:
#             IF_pickle=pkl.load(f)
#         IF_pickle_new=IF_pickle['influence']['identity'].to_numpy()
#         # print(IF_pickle_new)
#         summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
        
    
#         # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#         # print(summed_vector.shape)
#         # print(np.where(summed_vector>0)[1])
#         remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#         # print(np.sum(remaining_values))
#         # break
#         IFs.append(np.sum(summed_vector))
    
#     scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=0.12)
#     print(name)
#     print(list(scaled_numbers_fixed))


In [14]:
# import pickle as pkl

# with open(f'results_.25._mrpc.pkl', 'rb') as f:
#     IF_pickle=pkl.load(f)

In [15]:
# IF_pickle

In [16]:
# IF_pickle

In [6]:
with open(f'rebuttal/results_text_science_q_rebuttal_.5..pkl', 'rb') as f:
        IF_pickle=pkl.load(f)

In [7]:
IF_pickle

{'runtime': defaultdict(list, {'gauss_newton': 2494.553512096405}),
 'influence': defaultdict(list,
             {'gauss_newton':              0             1             2             3             4   \
              0    294.211609    310.180573    156.786377    184.958237    284.446899   
              1  -1570.005981  -1388.383179   -913.633911   -867.758362  -1355.923096   
              2   3525.703613   3857.236816   2235.309326   2282.262451   3925.561035   
              3     -7.557128    -46.585247      3.413703    -15.419071    -14.019368   
              4 -40659.074219 -50364.386719 -22190.689453 -24850.699219 -35448.695312   
              
                          5             6             7           8             9   ...  \
              0    12.880699    114.016922    263.692413    1.092401    212.234497  ...   
              1   -13.414119   -566.403076  -1308.924316   -9.387755  -1220.891846  ...   
              2    60.797092   1479.394897   3466.765869    7.

In [ ]:
#All IF values 

#names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']

names=['text_science_q_rebuttal']
for name in names:
    IFs = []
    for layer in layers:
        with open(f'rebuttal/results_{name}_{layer}.pkl', 'rb') as f:
            IF_pickle=pkl.load(f)
        IF_pickle_new=IF_pickle['influence']['gauss_newton'].to_numpy()
        # print(IF_pickle_new)
        summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
    
        # print(IF_pickle['influence']['proposed'].to_numpy().sum())
        # print(summed_vector.shape)
        # print(np.where(summed_vector>0)[1])
        # remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
        # print(np.sum(remaining_values))
        # break
        IFs.append(np.sum(summed_vector))
    
    scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2) #Can change target sum and exponent here
    print(name)
    print(list(scaled_numbers_fixed))


text_science_q_rebuttal
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 5, 5, 1, 6, 6, 5, 5, 5, 5, 5, 6, 5, 5, 5, 5, 5, 5]


In [ ]:
# #All_values

# #names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']

# names=['text_science_q_rebuttal']
# for name in names:
#     IFs = []
#     for layer in layers:
#         with open(f'rebuttal/results_{name}_{layer}.pkl', 'rb') as f:
#             IF_pickle=pkl.load(f)
#         IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
#         # print(IF_pickle_new)
#         summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
    
#         # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#         # print(summed_vector.shape)
#         # print(np.where(summed_vector>0)[1])
#         #remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#         # print(np.sum(remaining_values))
#         # break
#         IFs.append(np.sum(summed_vector))
    
#     scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2)
#     print(name)
#     print(list(scaled_numbers_fixed))


AttributeError: 'list' object has no attribute 'to_numpy'

In [ ]:
#Only positive IF Values

# IFs=[]
#names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']  #, 'commonq'
names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']
for name in names:
    IFs = []
    for layer in layers:
        with open(f'results_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle=pkl.load(f)
        IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
        summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
        # print(IF_pickle['influence']['proposed'].to_numpy().sum())
        # print(summed_vector.shape)
        # print(np.where(summed_vector>0)[1])
        remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
        # print(np.sum(remaining_values))
        # break
        IFs.append(np.sum(remaining_values))
    
    scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=3) #Can change target sum and exponent here
    print(name)
    print(list(scaled_numbers_fixed))


mrpc
[125, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]
cola
[110, 12, 5, 4, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
openbook
[129, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
text_scienceq
[129, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
commonq
[129, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [42]:
IFs

[-500759719559.22266,
 -836086789651.5234,
 -799352070805.1914,
 -846209671532.375,
 -815837596865.4219,
 -793182615698.1562,
 -870753830876.3516,
 -810871363271.0938,
 -909474679922.8369,
 -857012915215.9102,
 -844897002887.6914,
 -861992157758.5195,
 -814408108120.4219,
 -846907430870.8594,
 -818545914511.4766,
 -731334561582.0938,
 -825914795671.2344,
 -801113725401.0156,
 -780747281624.8008,
 -704670382205.4414,
 -793742463244.9258,
 -772368729137.0664,
 -693220715659.2383,
 -549356350514.6426,
 -657855035154.1641,
 -736495027131.0547,
 -645802436133.8164,
 -695086269755.1875,
 -537939230346.03906,
 -632287834816.2617,
 -684956925378.8008,
 -594755208309.3711]

In [ ]:
# #Only positive gemma

# # IFs=[]
# #names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']  #, 'commonq'
# names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']
# for name in names:
#     IFs = []
#     for layer in layers:
#         with open(f'results_gemma_{layer}_{name}.pkl', 'rb') as f:
#             IF_pickle=pkl.load(f)
#         IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
#         summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
#         # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#         # print(summed_vector.shape)
#         # print(np.where(summed_vector>0)[1])
#         remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#         # print(np.sum(remaining_values))
#         # break
#         IFs.append(np.sum(remaining_values))
    
#     scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2)
#     print(name)
#     print(list(scaled_numbers_fixed))


FileNotFoundError: [Errno 2] No such file or directory: 'results_gemma_.28._mrpc.pkl'

In [5]:
np.sort(IFs)

array([-1.05774413e+12, -1.05746613e+12, -1.05624660e+12, -1.05084797e+12,
       -1.03119337e+12, -1.00537460e+12, -9.73115013e+11, -9.71953135e+11,
       -9.64832104e+11, -9.60180280e+11, -9.59808553e+11, -9.28482647e+11,
       -9.28391144e+11, -9.27983594e+11, -9.18420619e+11, -8.98607439e+11,
       -8.97604832e+11, -8.82361731e+11, -8.75474679e+11, -8.54082961e+11,
       -8.30332849e+11, -8.30016383e+11, -8.20779647e+11, -7.50841023e+11,
       -7.33370659e+11, -6.84552765e+11, -6.79611706e+11, -6.66411110e+11,
       -6.23242146e+11, -5.92179476e+11, -5.24652222e+11, -5.22334209e+11])

In [4]:
np.argsort(IFs)

array([ 8,  9, 11,  7, 10, 13,  6, 16, 14, 12,  5,  2, 24, 20, 21, 15, 17,
        4,  1, 18,  3, 27, 25, 22, 19, 26,  0, 29, 23, 30, 28, 31])

In [ ]:
# #Only positive

# IFs=[]

# for layer in layers:
#     with open(f'results_{layer}_rte.pkl', 'rb') as f:
#         IF_pickle=pkl.load(f)
#     IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
#     summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
#     # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#     # print(summed_vector.shape)
#     # print(np.where(summed_vector>0)[1])
#     remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#     print(np.sum(remaining_values))
#     # break
#     IFs.append(np.sum(remaining_values))

In [7]:
np.argsort(np.abs(IFs))

array([31, 28, 30, 23, 29,  0, 26, 19, 22, 25, 27,  3, 18,  1,  4, 17, 15,
       21, 20, 24,  2,  5, 12, 14, 16,  6, 13, 10,  7, 11,  9,  8])

In [8]:
scale_values_final(IFs, 160)

array([1, 5, 6, 4, 5, 7, 7, 9, 7, 9, 9, 9, 7, 8, 7, 5, 7, 5, 5, 2, 6, 6,
       3, 1, 6, 4, 2, 4, 1, 1, 1, 1])

In [38]:
# Define the percentage of most negative values to keep (0.25, 0.50, or 0.75)
keep_ratio = 0.25 # Change this value as needed



names=['mrpc', 'cola', 'openbook', 'text_scienceq', 'commonq']
for name in names:
    IFs = []
    for layer in layers:
        with open(f'results_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle = pkl.load(f)
        
        IF_pickle_new = IF_pickle['influence']['proposed'].to_numpy()
        summed_vector = np.sum(IF_pickle_new, axis=0)
        
        # Get the threshold for the most negative values
        num_values_to_keep = int(len(summed_vector) * keep_ratio)
        threshold = np.partition(summed_vector, num_values_to_keep)[num_values_to_keep]
        
        # Keep only the most negative values below the threshold
        remaining_values = summed_vector[summed_vector <= threshold]
        # print(remaining_values[0:10])
        # print(len(remaining_values))
        
        # print(np.sum(remaining_values))
        IFs.append(np.sum(remaining_values))



    # Scale the values ensuring no zero values and preventing errors
    scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2.5)
    print(name)
    print(list(scaled_numbers_fixed))

mrpc
[4, 7, 8, 5, 6, 7, 7, 10, 11, 10, 8, 10, 6, 8, 7, 3, 7, 4, 3, 1, 6, 5, 1, 1, 5, 2, 1, 3, 1, 1, 1, 1]
cola
[1, 7, 9, 7, 8, 12, 11, 7, 10, 8, 8, 9, 6, 10, 7, 4, 7, 3, 2, 1, 3, 4, 2, 1, 6, 1, 1, 1, 1, 1, 1, 1]
openbook
[1, 7, 6, 12, 11, 7, 12, 6, 12, 10, 7, 6, 5, 5, 5, 7, 5, 4, 7, 3, 6, 3, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1]
text_scienceq
[1, 1, 1, 1, 1, 3, 4, 7, 3, 7, 6, 6, 6, 6, 8, 7, 7, 6, 6, 8, 7, 6, 6, 4, 5, 6, 6, 5, 4, 4, 8, 4]
commonq
[16, 12, 9, 10, 8, 6, 10, 5, 11, 8, 6, 8, 5, 6, 5, 2, 5, 4, 3, 1, 5, 4, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1]


In [ ]:
# 2.5 beta, 25% threshold

# mrpc
# [4, 7, 8, 5, 6, 7, 7, 10, 11, 10, 8, 10, 6, 8, 7, 3, 7, 4, 3, 1, 6, 5, 1, 1, 5, 2, 1, 3, 1, 1, 1, 1]
# cola
# [1, 7, 9, 7, 8, 12, 11, 7, 10, 8, 8, 9, 6, 10, 7, 4, 7, 3, 2, 1, 3, 4, 2, 1, 6, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 7, 6, 12, 11, 7, 12, 6, 12, 10, 7, 6, 5, 5, 5, 7, 5, 4, 7, 3, 6, 3, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1]
# text_scienceq
# [1, 1, 1, 1, 1, 3, 4, 7, 3, 7, 6, 6, 6, 6, 8, 7, 7, 6, 6, 8, 7, 6, 6, 4, 5, 6, 6, 5, 4, 4, 8, 4]
# commonq
# [16, 12, 9, 10, 8, 6, 10, 5, 11, 8, 6, 8, 5, 6, 5, 2, 5, 4, 3, 1, 5, 4, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 25% topk

# #mrpc
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

# commonq
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,1,2,1,1,1,1,1,1

# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,2,2,2,1,1,1,1

# textscienceq
# 1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1


In [ ]:
# 2.5 beta 50% threshold

# mrpc
# [1, 7, 7, 4, 5, 7, 8, 11, 8, 11, 9, 11, 6, 8, 7, 4, 7, 5, 3, 1, 6, 5, 1, 1, 6, 3, 1, 3, 1, 1, 1, 1]
# cola
# [1, 10, 9, 7, 8, 12, 10, 7, 10, 8, 8, 8, 6, 9, 7, 4, 7, 3, 2, 1, 3, 4, 2, 1, 6, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 6, 5, 10, 10, 7, 10, 6, 9, 9, 7, 6, 6, 6, 5, 7, 6, 5, 7, 4, 6, 4, 2, 1, 1, 3, 3, 3, 1, 1, 2, 1]
# text_scienceq
# [1, 1, 1, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 7, 7, 7, 6, 3, 7, 7, 6, 5, 5, 6, 7, 6, 5, 5, 7, 5]
# commonq
# [2, 11, 9, 11, 8, 7, 11, 6, 8, 9, 8, 9, 6, 7, 6, 2, 7, 5, 4, 1, 6, 4, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 50% topk

# #mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

# commonq
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,1,2,1,2,1,1,1,1

# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,2,2,2,1,1,1,1

# textscienceq
# 1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1

In [ ]:
# 2.5 beta 75% threshold

# mrpc
# [1, 5, 6, 3, 5, 7, 8, 11, 7, 12, 10, 11, 7, 9, 7, 4, 7, 5, 4, 1, 6, 5, 1, 1, 6, 3, 1, 3, 1, 1, 1, 1]
# cola
# [1, 7, 10, 7, 9, 12, 10, 7, 9, 9, 9, 8, 6, 9, 6, 4, 7, 3, 3, 2, 3, 4, 2, 1, 5, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 6, 5, 9, 9, 6, 9, 7, 8, 9, 7, 6, 6, 6, 6, 7, 6, 5, 7, 4, 6, 5, 3, 1, 1, 3, 3, 3, 1, 2, 2, 1]
# text_scienceq
# [1, 1, 1, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 6, 7, 7, 6, 4, 7, 6, 6, 5, 5, 6, 7, 6, 5, 6, 7, 5]
# commonq
# [1, 9, 7, 10, 8, 6, 11, 6, 9, 9, 8, 9, 6, 8, 7, 3, 7, 5, 5, 2, 6, 4, 2, 1, 1, 3, 1, 2, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 75% topk

#mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

#cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,1,1,1,1,1,1,1

#openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,2,2,1,2,2,1

#commonq
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,1,2,1,1,1,1

#text_science_q
# 1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

In [ ]:
#positive IF vals only

# mrpc
# 1,4,6,3,4,7,8,11,5,12,10,12,7,9,7,5,8,5,4,1,6,6,1,1,6,3,1,3,1,1,1,1
# cola
# 1,9,10,8,9,12,10,7,9,9,9,8,6,9,6,4,7,3,2,1,3,4,2,1,4,1,1,1,1,1,1,1
# openbook
# 1,5,5,8,7,5,8,7,8,5,7,7,7,6,6,6,6,6,7,5,7,5,3,2,2,4,3,4,2,2,3,1
# text_scienceq
# 1,1,2,1,2,2,4,6,4,6,6,6,6,6,7,6,7,7,6,3,7,6,6,5,5,6,7,6,5,6,7,5
# commonq
# 1,8,6,8,7,6,10,6,11,9,8,9,7,8,7,3,7,6,5,2,6,5,2,1,1,3,1,2,1,1,2,1

In [ ]:
#positive IF vals only topk

# mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1
# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1
# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1
# text_scienceq
# 1,1,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
# commonq
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,1,2,1,1,2,1

In [ ]:
# mrpc
# [1, 4, 5, 2, 4, 7, 8, 13, 13, 13, 11, 13, 7, 10, 7, 4, 8, 4, 3, 1, 5, 5, 1, 1, 5, 2, 1, 2]
# cola
# [1, 15, 10, 7, 9, 12, 11, 7, 9, 8, 9, 7, 6, 9, 6, 3, 7, 3, 2, 2, 3, 4, 2, 1, 4, 1, 1, 1]
# openbook
# [1, 5, 5, 8, 8, 6, 8, 7, 8, 9, 8, 7, 7, 6, 6, 6, 6, 6, 7, 5, 7, 5, 3, 2, 2, 4, 4, 4]
# text_scienceq
# [1, 2, 2, 1, 2, 3, 4, 6, 4, 7, 7, 6, 7, 7, 8, 7, 8, 8, 7, 8, 8, 7, 7, 6, 6, 6, 8, 7]
# commonq
# [1, 8, 6, 8, 7, 6, 10, 6, 13, 9, 8, 9, 7, 8, 7, 3, 7, 6, 5, 3, 6, 5, 2, 1, 2, 3, 2, 2]

In [10]:
# Check if the same indexes have 1s in both rows1, 4, 5, 2, 4, 7, 8, 13, 13, 13, 11, 13, 7, 10, 7, 4, 8, 4, 3, 1, 5, 5, 1, 1, 5, 2, 1, 2
number_experts = [5,5,5,5,5,5,5,5,5,5,5,5,5,5,6,5,5,1,6,6,5,5,5,5,5,6,5,5,5,5,5,5]
# top_k = [1, 2, 2, 1, 2, 3, 4, 6, 4, 7, 7, 6, 7, 7, 81,2,2,1,2,3,4,6,4,7,7,6,7,7,8,7,8,8,7,8,8,7,7,6,6,6,8,7, 7, 8, 8, 7, 8, 8, 7, 7, 6, 6, 6, 8, 7]

# Find indexes where top_k has 1s
indexes_with_ones = [i for i, val in enumerate(number_experts) if val == 1]

# Check if the same indexes have 1s in number_experts
top_k=[]
for i in range(len(number_experts)):
    if number_experts[i]==1:
        top_k.append(1)
    else:
        top_k.append(2)
print(top_k)



[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [25]:
scaled_numbers_fixed  #rte

array([16, 12,  9, 10,  8,  6, 10,  5, 11,  8,  6,  8,  5,  6,  5,  2,  5,
        4,  3,  1,  5,  4,  1,  1,  1,  2,  1,  1,  1,  1,  1,  1])

In [ ]:
scaled_numbers_fixed  #mrpc

array([ 1,  4,  6,  3,  4,  7,  8, 11,  5, 12, 10, 12,  7,  9,  7,  5,  8,
        5,  4,  1,  6,  6,  1,  1,  6,  3,  1,  3,  1,  1,  1,  1])

In [ ]:
scaled_numbers_fixed #cola

array([ 1,  9, 10,  8,  9, 12, 10,  7,  9,  9,  9,  8,  6,  9,  6,  4,  7,
        3,  2,  1,  3,  4,  2,  1,  4,  1,  1,  1,  1,  1,  1,  1])

In [ ]:
scaled_numbers_fixed  #openbook

array([1, 5, 4, 8, 8, 5, 8, 7, 8, 6, 7, 7, 7, 6, 6, 6, 6, 6, 7, 5, 7, 5,
       3, 2, 2, 4, 3, 4, 2, 2, 2, 1])

In [17]:
scaled_numbers_fixed  #commonq

array([ 1,  8,  6,  9,  7,  6, 10,  7,  7,  9,  8, 10,  7,  9,  7,  3,  7,
        6,  5,  2,  6,  5,  2,  1,  1,  3,  1,  2,  1,  1,  2,  1])

In [ ]:
scaled_numbers_fixed  #text_science_q

array([1, 1, 1, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 7, 7, 7, 6, 3, 7, 6,
       6, 5, 5, 6, 7, 6, 5, 6, 7, 5])

In [57]:
scaled_numbers_fixed  #cola

array([ 1,  9, 10,  8,  9, 12, 10,  7,  9,  9,  9,  8,  6,  9,  6,  4,  7,
        3,  2,  1,  3,  4,  2,  1,  4,  1,  1,  1,  1,  1,  1,  1])

: 

In [45]:
scaled_numbers_fixed   #openbook

array([1, 5, 5, 8, 7, 5, 8, 7, 8, 5, 7, 7, 7, 6, 6, 6, 6, 6, 7, 5, 7, 5,
       3, 2, 2, 4, 3, 4, 2, 2, 3, 1])

In [38]:
scaled_numbers_fixed  #rte

array([1, 1, 3, 5, 4, 5, 8, 8, 8, 8, 9, 8, 8, 7, 7, 7, 6, 7, 6, 4, 4, 7,
       4, 3, 6, 4, 2, 3, 1, 3, 2, 1])

In [31]:
scaled_numbers_fixed  #mrpc

array([ 1,  4,  6,  3,  4,  7,  8, 11,  5, 12, 10, 12,  7,  9,  7,  5,  8,
        5,  4,  1,  6,  6,  1,  1,  6,  3,  1,  3,  1,  1,  1,  1])

In [24]:
scaled_numbers_fixed  #commonq_new

array([ 1,  8,  6,  8,  7,  6, 10,  6, 11,  9,  8,  9,  7,  8,  7,  3,  7,
        6,  5,  2,  6,  5,  2,  1,  1,  3,  1,  2,  1,  1,  2,  1])

In [17]:
scaled_numbers_fixed #text_science_q_new

array([1, 1, 2, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 6, 7, 7, 6, 3, 7, 6,
       6, 5, 5, 6, 7, 6, 5, 6, 7, 5])

In [ ]:
# expert number:  1,3,5,4,5,5,4,4,3,4,3,2,2,3,3,4,9,4,7,7,7,7,7,7,9,7,6,8,6,7,4,3   alphalora
# top_k:  1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

In [46]:
scaled_numbers_fixed ##text_science_q-10

# expert number:  1,3,5,4,5,5,4,4,3,4,3,2,2,3,3,4,9,4,7,7,7,7,7,7,9,7,6,8,6,7,4,3   alphalora

array([1, 3, 4, 2, 4, 4, 5, 6, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5,
       5, 5, 4, 5, 6, 5, 4, 5, 6, 4])

: 

In [33]:
scaled_numbers_fixed ##rte-100/10

array([1, 1, 3, 3, 4, 4, 6, 6, 5, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 5, 5, 6,
       5, 6, 6, 7, 5, 5, 4, 5, 5, 4])

In [24]:
scaled_numbers_fixed ###openbook-100/10

array([1, 5, 5, 6, 7, 5, 6, 6, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 5,
       5, 4, 3, 5, 5, 5, 3, 4, 4, 2])

In [17]:
scaled_numbers_fixed ###commonq-100/10

array([ 2,  5,  3,  7,  4,  5,  7,  6, 10,  7,  7,  8,  6,  8,  8,  4,  8,
        6,  5,  3,  6,  7,  4,  1,  3,  6,  2,  4,  1,  2,  3,  2])

In [ ]:
scaled_numbers_fixed ###cola-100/10

In [11]:
scaled_numbers_fixed.sum()

160

In [1]:
import os
import re

# Directory containing the files
directory = "/nas02/Hadi/Model-Selection-IF/alphalora/DataInf/src"

# Regex pattern to match files (N can be 0-32, X can be specified values or ".")
pattern = re.compile(r"results_\.(\d{1,2})\._(cola|rte|openbook|commonq|text_scienceq|\.)\.pkl")

# Define new naming format (modify as needed)
def new_name(old_name, match):
    N = match.group(1)  # Extract the number (0-32)
    X = match.group(2)  # Extract the dataset name or "."
    
    # Example renaming: Change "results_.N._X.pkl" to "renamed_N_X.pkl"
    new_filename = f"renamed_{N}_{X}.pkl"  # Replace "." with "dot" for clarity
    return new_filename

# Iterate through files and rename them
for filename in os.listdir(directory):
    match = pattern.match(filename)
    if match:
        new_filename = new_name(filename, match)
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)

        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")


Renamed: results_.0._cola.pkl -> renamed_0_cola.pkl
Renamed: results_.1._text_scienceq.pkl -> renamed_1_text_scienceq.pkl
Renamed: results_.17._cola.pkl -> renamed_17_cola.pkl
Renamed: results_.25._text_scienceq.pkl -> renamed_25_text_scienceq.pkl
Renamed: results_.0._text_scienceq.pkl -> renamed_0_text_scienceq.pkl
Renamed: results_.11._text_scienceq.pkl -> renamed_11_text_scienceq.pkl
Renamed: results_.26._rte.pkl -> renamed_26_rte.pkl
Renamed: results_.20._cola.pkl -> renamed_20_cola.pkl
Renamed: results_.16._cola.pkl -> renamed_16_cola.pkl
Renamed: results_.13._rte.pkl -> renamed_13_rte.pkl
Renamed: results_.23._text_scienceq.pkl -> renamed_23_text_scienceq.pkl
Renamed: results_.6._commonq.pkl -> renamed_6_commonq.pkl
Renamed: results_.25._openbook.pkl -> renamed_25_openbook.pkl
Renamed: results_.18._commonq.pkl -> renamed_18_commonq.pkl
Renamed: results_.10._text_scienceq.pkl -> renamed_10_text_scienceq.pkl
Renamed: results_.18._cola.pkl -> renamed_18_cola.pkl
Renamed: results_.24

In [2]:
import os
import re

# Directory containing the files
directory = "/nas02/Hadi/Model-Selection-IF/alphalora/DataInf/src"

# Regex pattern to match files (N can be 0-32)
pattern = re.compile(r"results_\.(\d{1,2})\.\.pkl")

# Function to generate new filename
def new_name(old_name, match):
    N = match.group(1)  # Extract the number (0-32)
    return f"renamed_{N}_mrpc.pkl"  # New format

# Iterate through files and rename them
for filename in os.listdir(directory):
    match = pattern.match(filename)
    if match:
        new_filename = new_name(filename, match)
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)

        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")


Renamed: results_.10..pkl -> renamed_10_mrpc.pkl
Renamed: results_.7..pkl -> renamed_7_mrpc.pkl
Renamed: results_.27..pkl -> renamed_27_mrpc.pkl
Renamed: results_.1..pkl -> renamed_1_mrpc.pkl
Renamed: results_.2..pkl -> renamed_2_mrpc.pkl
Renamed: results_.12..pkl -> renamed_12_mrpc.pkl
Renamed: results_.11..pkl -> renamed_11_mrpc.pkl
Renamed: results_.23..pkl -> renamed_23_mrpc.pkl
Renamed: results_.8..pkl -> renamed_8_mrpc.pkl
Renamed: results_.3..pkl -> renamed_3_mrpc.pkl
Renamed: results_.18..pkl -> renamed_18_mrpc.pkl
Renamed: results_.21..pkl -> renamed_21_mrpc.pkl
Renamed: results_.26..pkl -> renamed_26_mrpc.pkl
Renamed: results_.19..pkl -> renamed_19_mrpc.pkl
Renamed: results_.28..pkl -> renamed_28_mrpc.pkl
Renamed: results_.22..pkl -> renamed_22_mrpc.pkl
Renamed: results_.9..pkl -> renamed_9_mrpc.pkl
Renamed: results_.14..pkl -> renamed_14_mrpc.pkl
Renamed: results_.20..pkl -> renamed_20_mrpc.pkl
Renamed: results_.15..pkl -> renamed_15_mrpc.pkl
Renamed: results_.0..pkl -> rena

In [2]:
with open(f'results_.31._commonq.pkl', 'rb') as f:
    IF_pickle=pkl.load(f)

In [3]:
IF_pickle.keys()

dict_keys(['runtime', 'influence'])

In [4]:
IF_pickle['influence']['proposed']

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-22090398.0,-24105104.00,-3.184808e+07,-2.656872e+07,-2.462000e+07,-3.130805e+07,-3.017327e+07,-26148530.00,-22875330.00,-3.166971e+07,...,-2.447012e+07,-3.070553e+07,-1.907009e+07,-25991564.00,-2.898626e+07,-2.790913e+07,-2.687778e+07,-3.597925e+07,-2.417830e+07,-1.861745e+07
1,-25332608.0,-49746124.00,-4.678189e+07,-3.945274e+07,-3.882874e+07,-7.570066e+07,-4.324963e+07,-42875472.00,-55239704.00,-6.268493e+07,...,-2.132023e+08,-5.930026e+07,-5.704939e+07,-49676952.00,-5.800691e+07,-6.944664e+07,-5.690476e+07,-7.029039e+07,-7.550058e+07,-9.865981e+07
2,-37703472.0,-57895432.00,-6.633182e+07,-6.291757e+07,-4.831100e+07,-8.094491e+07,-6.178739e+07,-53380476.00,-48358832.00,-7.773998e+07,...,-5.946184e+07,-7.459075e+07,-5.308262e+07,-64174064.00,-6.850939e+07,-7.268498e+07,-7.069770e+07,-8.615821e+07,-5.706033e+07,-6.085111e+07
3,-21759062.0,-30972478.00,-3.840629e+07,-3.052700e+07,-2.886810e+07,-4.432324e+07,-3.232820e+07,-30553194.00,-34814352.00,-4.092177e+07,...,-4.699730e+07,-3.759482e+07,-4.254547e+07,-39617888.00,-4.263676e+07,-4.948694e+07,-4.068566e+07,-4.723512e+07,-4.000893e+07,-4.324133e+07
4,-33884432.0,-51820588.00,-5.791220e+07,-4.440004e+07,-3.674674e+07,-4.688598e+07,-4.186814e+07,-35351388.00,-32983196.00,-5.034225e+07,...,-2.936695e+07,-4.355528e+07,-3.883090e+07,-45216240.00,-4.567202e+07,-4.606120e+07,-3.978302e+07,-4.834854e+07,-3.546362e+07,-4.002582e+07
5,-26528426.0,-35214032.00,-4.123464e+07,-4.770946e+07,-4.218762e+07,-6.485198e+07,-4.085525e+07,-36553824.00,-35230872.00,-4.803096e+07,...,-5.108574e+07,-4.844262e+07,-4.109762e+07,-43832184.00,-4.949246e+07,-4.599255e+07,-5.376176e+07,-5.877808e+07,-5.376692e+07,-4.241760e+07
6,-26374776.0,-33129384.00,-4.481393e+07,-3.742520e+07,-2.579624e+07,-3.755326e+07,-3.342560e+07,-26852380.00,-33447884.00,-3.190585e+07,...,-1.318523e+07,-3.547458e+07,-2.803795e+07,-33005228.00,-3.660304e+07,-3.872036e+07,-2.968453e+07,-3.557928e+07,-2.451791e+07,-2.341794e+07
7,-32066476.0,-39060900.00,-3.668117e+07,-4.495309e+07,-4.189391e+07,-6.034968e+07,-4.283491e+07,-35086704.00,-39270300.00,-5.073826e+07,...,-4.692570e+07,-5.255322e+07,-4.998497e+07,-43670224.00,-6.172979e+07,-4.747929e+07,-5.242746e+07,-5.802289e+07,-4.750402e+07,-5.468132e+07
8,-36545820.0,-50036856.00,-5.243380e+07,-5.240581e+07,-4.818869e+07,-6.602882e+07,-4.689722e+07,-47123236.00,-36226520.00,-6.142009e+07,...,-1.028686e+08,-6.191796e+07,-4.864910e+07,-42904536.00,-5.540020e+07,-5.010634e+07,-5.207370e+07,-6.579707e+07,-4.987626e+07,-6.949742e+07
9,-34387040.0,-40662168.00,-5.178664e+07,-4.650709e+07,-3.926392e+07,-7.358787e+07,-4.671326e+07,-35208608.00,-50211328.00,-5.177139e+07,...,-4.645077e+07,-5.753479e+07,-6.886290e+07,-49308096.00,-6.727019e+07,-6.278206e+07,-6.082948e+07,-6.175100e+07,-5.317814e+07,-6.456156e+07
